# 31 — Calibration Analysis

Reliability diagram: when the model says 70%, does YES win 70% of the time? Compare model calibration vs market calibration.

In [ ]:
import sys
sys.path.insert(0, '/Users/chriswang/Desktop/prediction-market-exploration/nba-edge')

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from nba_edge.data.s3_reader import discover_trade_dates, load_trades, load_boxscores_for_game, discover_game_ids
from nba_edge.data.ticker_parser import parse_ticker, is_game_winner
from nba_edge.data.game_alignment import match_ticker_to_game
from nba_edge.backtest.engine import BacktestEngine
from nba_edge.backtest.metrics import calibration_bins
from nba_edge.models.analytical import AnalyticalWinProb

In [ ]:
# Run backtest (same setup)
dates = discover_trade_dates(min_date='2026-04-21')
df = load_trades(dates)
engine = BacktestEngine(model=AnalyticalWinProb(variance_rate=0.44))
game_tickers = [t for t in df['market_ticker'].unique().to_list() if is_game_winner(t)]

results = []
for ticker in sorted(game_tickers):
    parsed = parse_ticker(ticker)
    if not parsed.game_date:
        continue
    try:
        available = discover_game_ids(parsed.game_date)
    except Exception:
        continue
    matched = match_ticker_to_game(ticker, available)
    if not matched:
        continue
    snapshots = load_boxscores_for_game(matched, parsed.game_date)
    if len(snapshots) < 50 or snapshots[-1].get('game_status', 0) != 3:
        continue
    ticker_trades = df.filter(pl.col('market_ticker') == ticker)
    result = engine.run_game(ticker_trades, snapshots, ticker)
    if result and len(result.trades) > 0:
        results.append(result)

# Extract arrays
model_probs = [t.model_prob_yes for r in results for t in r.trades]
market_probs = [t.market_implied for r in results for t in r.trades]
outcomes = [r.outcome_yes for r in results for t in r.trades]
print(f'{len(model_probs):,} observations from {len(results)} markets')

In [ ]:
# Calibration bins for both model and market
model_cal = calibration_bins(model_probs, outcomes, n_bins=10)
market_cal = calibration_bins(market_probs, outcomes, n_bins=10)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect')

# Model
m_pred = [b['predicted_mean'] for b in model_cal]
m_act = [b['actual_mean'] for b in model_cal]
m_n = [b['count'] for b in model_cal]
ax.scatter(m_pred, m_act, s=[n/50 for n in m_n], color='red', alpha=0.7, label='Model', zorder=3)
ax.plot(m_pred, m_act, color='red', alpha=0.4)

# Market
mk_pred = [b['predicted_mean'] for b in market_cal]
mk_act = [b['actual_mean'] for b in market_cal]
mk_n = [b['count'] for b in market_cal]
ax.scatter(mk_pred, mk_act, s=[n/50 for n in mk_n], color='blue', alpha=0.7, label='Market', zorder=3)
ax.plot(mk_pred, mk_act, color='blue', alpha=0.4)

ax.set_xlabel('Predicted Probability')
ax.set_ylabel('Actual Outcome Rate')
ax.set_title('Calibration: Model vs Market')
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Calibration error (ECE — Expected Calibration Error)
def ece(bins):
    total = sum(b['count'] for b in bins)
    return sum(abs(b['predicted_mean'] - b['actual_mean']) * b['count'] / total for b in bins)

print(f'Expected Calibration Error:')
print(f'  Model:  {ece(model_cal):.4f}')
print(f'  Market: {ece(market_cal):.4f}')
print(f'  (Lower = better calibrated)')

In [ ]:
# Detailed bin table
print(f"{'Bin':>5s} {'Model Pred':>11s} {'Model Act':>10s} {'Mkt Pred':>10s} {'Mkt Act':>9s} {'N (model)':>10s}")
print('-' * 60)
for mb, mkb in zip(model_cal, market_cal):
    print(f"{mb['bin_center']:>5.2f} {mb['predicted_mean']:>11.3f} {mb['actual_mean']:>10.3f} "
          f"{mkb['predicted_mean']:>10.3f} {mkb['actual_mean']:>9.3f} {mb['count']:>10,}")

In [ ]:
# Where is the model miscalibrated?
# Check if model is systematically over/under-confident
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Model residuals by predicted prob
model_arr = np.array(model_probs)
outcome_arr = np.array(outcomes, dtype=float)
residuals = model_arr - outcome_arr  # positive = overconfident in YES

# Bin residuals
for i, (lo, hi) in enumerate(zip(np.arange(0, 1, 0.1), np.arange(0.1, 1.1, 0.1))):
    mask = (model_arr >= lo) & (model_arr < hi)
    if mask.sum() > 0:
        axes[0].boxplot(residuals[mask], positions=[i], widths=0.6, showfliers=False)

axes[0].set_xticklabels([f'{x:.0%}' for x in np.arange(0.05, 1, 0.1)])
axes[0].axhline(0, color='black', linestyle=':')
axes[0].set_xlabel('Model Predicted P(yes)')
axes[0].set_ylabel('Residual (predicted - outcome)')
axes[0].set_title('Model Residuals by Prediction Bin')

# Histogram of model - market differences
diff = model_arr - np.array(market_probs)
axes[1].hist(diff * 100, bins=50, edgecolor='none', alpha=0.7)
axes[1].axvline(0, color='black', linestyle='-')
axes[1].set_xlabel('Model - Market (cents)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Model vs Market Disagreement\n(mean={np.mean(diff)*100:.2f}c, std={np.std(diff)*100:.2f}c)')

plt.tight_layout()
plt.show()